In [1]:
import os
import re
import time
import unicodedata
from difflib import SequenceMatcher

import pandas as pd
import musicbrainzngs
import networkx as nx

No need to rerun unless adding a new csv or lowering the N. Can manually add the top people that were missed

In [ ]:
import os
import re
import time
import unicodedata
from difflib import SequenceMatcher

import pandas as pd
import musicbrainzngs

musicbrainzngs.set_useragent(
    app='BillboardFeatureCrawler',
    version='1.0',
    contact='your-email@example.com'
)

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

HOT100_FILE = 'hot100.csv'

MBID_CACHE_FILE = os.path.join(DATA_DIR, 'mbid_cache.csv') 
OUTPUT_MATCHED = os.path.join(DATA_DIR, 'billboard_to_musicbrainz.csv')
MATCH_REPORT = os.path.join(DATA_DIR, 'billboard_name_match_report.csv')
NON_PERFECT_REPORT = os.path.join(DATA_DIR, 'billboard_nonperfect_matches.csv')
UNMATCHED_REPORT = os.path.join(DATA_DIR, 'billboard_unmatched.csv')

def normalize_name(name):
    if pd.isna(name):
        return ''
    text = unicodedata.normalize('NFKD', str(name)).encode('ascii', 'ignore').decode('ascii')
    text = text.lower().strip()
    text = text.replace('&', ' and ')
    text = re.sub(r'\(.*?\)|\[.*?\]', ' ', text)
    text = re.sub(r'\b(feat|featuring|ft)\.?\b.*$', ' ', text)
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def strip_the_prefix(name):
    return re.sub(r'^the\s+', '', name).strip()

def fetch_mbid_for_name(name):
    try:

        result = musicbrainzngs.search_artists(artist=name, limit=1)
        
        
        if 'artist-list' in result and result['artist-list']:
            artist = result['artist-list'][0]
            return {
                'mbid': artist['id'],
                'artist_mb': artist['name'],
                'type': artist.get('type', 'Unknown'),
                'country_mb': artist.get('country', 'Unknown')
            }
    except Exception as e:
        print(f"    [!] Error fetching '{name}': {e}")
    return None

hot100_df = pd.read_csv(HOT100_FILE, low_memory=False)
hot100_df.columns = hot100_df.columns.str.strip()

hot100_df = hot100_df.dropna(subset=['Artist']).copy()

artist_counts = hot100_df['Artist'].value_counts().reset_index()
artist_counts.columns = ['Artist', 'Chart_Appearances']
unique_artists = artist_counts['Artist'].unique()

candidate_data = []
existing_names = set()

if os.path.exists(MBID_CACHE_FILE) and os.path.getsize(MBID_CACHE_FILE) > 0:
    try:
        cache_df = pd.read_csv(MBID_CACHE_FILE)
        if 'billboard_name' in cache_df.columns:
            candidate_data = cache_df.to_dict('records')
            existing_names = set(cache_df['billboard_name'])
            print(f"   -> Loaded {len(existing_names)} artists from local cache.")
    except pd.errors.EmptyDataError:
        print("   -> Cache file was empty or corrupted. Starting fresh.")

new_fetches = 0
for idx, name in enumerate(unique_artists):
    if name in existing_names:
        continue
    
    if new_fetches % 50 == 0 and new_fetches > 0:
        print(f"   -> Fetched {new_fetches} new artists from API...")
        
    info = fetch_mbid_for_name(name)
    if info:
        info['billboard_name'] = name
        candidate_data.append(info)
    else:
        candidate_data.append({'billboard_name': name, 'mbid': None, 'artist_mb': None})
    
    pd.DataFrame(candidate_data).to_csv(MBID_CACHE_FILE, index=False)
    time.sleep(1.05) 
    new_fetches += 1

candidates_df = pd.DataFrame(candidate_data)

candidates = candidates_df.dropna(subset=['artist_mb']).copy()
candidates['candidate_name'] = candidates['artist_mb'].astype(str).str.strip()
candidates = candidates[candidates['candidate_name'] != ''].copy()

candidates['norm'] = candidates['candidate_name'].apply(normalize_name)
candidates['norm_no_the'] = candidates['norm'].apply(strip_the_prefix)
candidates = candidates[candidates['norm'] != ''].copy()

best_by_norm = candidates.drop_duplicates(subset=['norm']).set_index('norm')
best_by_norm_no_the = (
    candidates[candidates['norm_no_the'] != '']
    .drop_duplicates(subset=['norm_no_the'])
    .set_index('norm_no_the')
)

bucket_by_first_char = {}
for norm_value in best_by_norm.index:
    if not norm_value: continue
    first_char = norm_value[0]
    bucket_by_first_char.setdefault(first_char, []).append(norm_value)

rows = []
for _, row_data in artist_counts.iterrows():
    listener_name = str(row_data['Artist']).strip()
    chart_appearances = row_data['Chart_Appearances']
    
    if listener_name == '':
        continue

    norm = normalize_name(listener_name)
    norm_no_the = strip_the_prefix(norm)

    matched = None
    match_type = 'unmatched'
    match_score = 0.0

    if norm in best_by_norm.index:
        matched = best_by_norm.loc[norm]
        matched_name_norm = normalize_name(matched['candidate_name'])
        raw_norm = normalize_name(listener_name)
        match_type = 'exact_text' if raw_norm == matched_name_norm else 'exact_normalized'
        match_score = 1.0
    elif norm_no_the in best_by_norm_no_the.index:
        matched = best_by_norm_no_the.loc[norm_no_the]
        match_type = 'exact_normalized'
        match_score = 1.0
    else:
        if norm:
            first_char = norm[0]
            candidates_in_bucket = bucket_by_first_char.get(first_char, [])
            narrowed = [x for x in candidates_in_bucket if abs(len(x) - len(norm)) <= 4]
            search_space = narrowed if narrowed else candidates_in_bucket

            best_norm = None
            best_score = 0.0
            for candidate_norm in search_space:
                score = SequenceMatcher(None, norm, candidate_norm).ratio()
                if score > best_score:
                    best_score = score
                    best_norm = candidate_norm

            if best_norm is not None and best_score >= 0.90:
                matched = best_by_norm.loc[best_norm]
                match_type = 'fuzzy'
                match_score = round(best_score, 4)

    row_out = {
        'listener_artist': listener_name,
        'chart_appearances': chart_appearances,
        'match_type': match_type,
        'match_score': match_score,
    }

    if matched is not None:
        row_out.update({
            'mbid': matched['mbid'],
            'artist_mb': matched['artist_mb'],
            'matched_name': matched['candidate_name'],
        })
    else:
        row_out.update({
            'mbid': pd.NA,
            'artist_mb': pd.NA,
            'matched_name': pd.NA,
        })

    rows.append(row_out)

match_df = pd.DataFrame(rows)

match_df = match_df.sort_values(by='chart_appearances', ascending=False, na_position='last')

matched_df = match_df.dropna(subset=['mbid']).copy()
matched_df = matched_df.drop_duplicates(subset=['mbid'], keep='first')

match_df.to_csv(MATCH_REPORT, index=False)
match_df[match_df['match_type'] != 'exact_text'].to_csv(NON_PERFECT_REPORT, index=False)
match_df[match_df['match_type'] == 'unmatched'].to_csv(UNMATCHED_REPORT, index=False)
matched_df.to_csv(OUTPUT_MATCHED, index=False)

print('\n--- SUMMARY ---')
print(f"Total Unique Billboard Artists: {len(unique_artists):,}")
print(f"Matched artists with MBIDs: {len(matched_df):,}")
print(f"Unmatched artists: {(match_df['match_type'] == 'unmatched').sum():,}")
print(f"-> Successfully saved crawl input file to: {OUTPUT_MATCHED}")

Now we use our list of artists to form a edge csv.

In [ ]:
import os
import time
import pandas as pd
import musicbrainzngs


musicbrainzngs.set_useragent(
    app='BillboardFeatureCrawler',
    version='2.0',
    contact='your-email@example.com'
)

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

INPUT_CSV = os.path.join(DATA_DIR, 'billboard_to_musicbrainz.csv')

EDGES_FILE = os.path.join(DATA_DIR, 'mbz_feature_edges_billboard_only.csv') 
PROCESSED_FILE = os.path.join(DATA_DIR, 'mbz_processed_artists_billboard.txt')

HISTORICAL_DB_PATH = r"C:\School\Networks ORF 387\Git_new\Spotify-Networks\Folder\mbz_feature_edges_detailed.csv"

SAVE_EVERY = 10
PAGE_SIZE = 100
MAX_RECORDINGS_PER_ARTIST = 500

def save_state(edge_rows, processed_mbids):
    if edge_rows:
        pd.DataFrame(edge_rows).drop_duplicates().to_csv(EDGES_FILE, index=False)
    with open(PROCESSED_FILE, 'w', encoding='utf-8') as f:
        f.write('\n'.join(sorted(processed_mbids)))

print('Loading matched artist seed list...')

df = pd.read_csv(INPUT_CSV).dropna(subset=['mbid', 'artist_mb']).copy()
df['mbid'] = df['mbid'].astype(str).str.strip()
df['artist_mb'] = df['artist_mb'].astype(str).str.strip()
df = df[df['mbid'] != '']

edges = []
completed_artists = set()
edge_keys = set()

if os.path.exists(EDGES_FILE):
    print('Found existing current edge file. Loading progress...')
    previous_edges = pd.read_csv(EDGES_FILE)
    edges = previous_edges.to_dict('records')

    if {'Source_MBID', 'Target_MBID', 'Recording_MBID'}.issubset(previous_edges.columns):
        for _, r in previous_edges.iterrows():
            edge_keys.add((
                str(r.get('Source_MBID', '')).strip(),
                str(r.get('Target_MBID', '')).strip(),
                str(r.get('Recording_MBID', '')).strip(),
            ))

if os.path.exists(PROCESSED_FILE):
    with open(PROCESSED_FILE, 'r', encoding='utf-8') as f:
        completed_artists = set(line.strip() for line in f.readlines() if line.strip())
    print(f'Skipping {len(completed_artists)} already processed artists.')

historical_edges_dict = {}
if os.path.exists(HISTORICAL_DB_PATH):
    print(f'Loading historical database from {HISTORICAL_DB_PATH} to save API calls...')
    hist_df = pd.read_csv(HISTORICAL_DB_PATH, low_memory=False)

    for source_mbid, group in hist_df.groupby('Source_MBID'):
        historical_edges_dict[str(source_mbid).strip()] = group.to_dict('records')
    print(f'-> Historical DB loaded. Contains data for {len(historical_edges_dict)} unique source artists.')
else:
    print(f'[!] Historical DB not found at {HISTORICAL_DB_PATH}. Will use API for all.')

artists_to_process = df[~df['mbid'].isin(completed_artists)]
print(f"\nStarting crawl. {len(artists_to_process)} artists remaining...\n")

try:
    for idx, (_, row) in enumerate(artists_to_process.iterrows(), start=1):
        source_mbid = row['mbid']
        source_name = row['artist_mb']
        print(f'[{idx}/{len(artists_to_process)}] Mining: {source_name}')

        if source_mbid in historical_edges_dict:
            hist_records = historical_edges_dict[source_mbid]
            added_count = 0

            for r in hist_records:
                target_mbid = str(r['Target_MBID']).strip()
                key = (str(r['Source_MBID']).strip(), target_mbid, str(r['Recording_MBID']).strip())
                
                if key not in edge_keys:
                    edge_keys.add(key)
                    edges.append(r)
                    added_count += 1
            
            print(f'   -> Found {added_count} edges in historical database. Skipping API.')
            completed_artists.add(source_mbid)
            
            if len(completed_artists) % SAVE_EVERY == 0:
                save_state(edges, completed_artists)
            continue 

        collab_count = 0
        fetched = 0
        offset = 0

        try:
            while fetched < MAX_RECORDINGS_PER_ARTIST:
                response = musicbrainzngs.browse_recordings(
                    artist=source_mbid,
                    includes=['artist-credits'],
                    limit=PAGE_SIZE,
                    offset=offset,
                )
                recording_list = response.get('recording-list', [])
                if not recording_list:
                    break

                for recording in recording_list:
                    recording_id = str(recording.get('id', '')).strip()
                    recording_title = str(recording.get('title', '')).strip()
                    credits = recording.get('artist-credit', [])

                    if not isinstance(credits, list):
                        continue

                    collaborators = []
                    for credit in credits:
                        if isinstance(credit, dict) and 'artist' in credit:
                            artist_obj = credit['artist']
                            collaborator_id = str(artist_obj.get('id', '')).strip()
                            collaborator_name = str(artist_obj.get('name', '')).strip()
                            if collaborator_id and collaborator_name:
                                collaborators.append((collaborator_id, collaborator_name))

                    if len(collaborators) <= 1:
                        continue

                    has_source = any(c_id == source_mbid for c_id, _ in collaborators)
                    if not has_source:
                        continue

                    for target_mbid, target_name in collaborators:
                        if target_mbid == source_mbid:
                            continue

                        key = (source_mbid, target_mbid, recording_id)
                        if key in edge_keys:
                            continue

                        edge_keys.add(key)
                        edges.append({
                            'Source_MBID': source_mbid,
                            'Source_Name': source_name,
                            'Target_MBID': target_mbid,
                            'Target_Name': target_name,
                            'Recording_MBID': recording_id,
                            'Track_Name': recording_title,
                            'Relation_Type': 'artist_credit_feature',
                        })
                        collab_count += 1

                fetched += len(recording_list)
                offset += PAGE_SIZE
                time.sleep(1.05)

                if len(recording_list) < PAGE_SIZE:
                    break

            print(f'   -> Fetched {collab_count} edges from API.')
            completed_artists.add(source_mbid)

            if len(completed_artists) % SAVE_EVERY == 0:
                save_state(edges, completed_artists)

        except musicbrainzngs.ResponseError as e:
            print(f'   [!] MusicBrainz API Error on {source_name}: {e}')
            completed_artists.add(source_mbid)
            continue

except KeyboardInterrupt:
    print('\n[!] Crawl paused by user. Saving state...')
except Exception as e:
    print(f'\n[!] Unexpected crash: {e}. Saving state...')
finally:
    save_state(edges, completed_artists)
    print('>>> Save State Updated. Safe to close. <<<')